# Monte Carlo π with nb2slurm

A complete, runnable nb2slurm example. The workflow estimates π by Monte Carlo:
throw random darts at the unit square and count how many land inside the quarter
circle. We run it for several sample sizes and random seeds — each
`(n_samples, seed)` pair is one SLURM job.

This control notebook builds the scripts, shows how to test the chain locally,
and then syncs everything to an HPC and runs it there. The workflow notebooks
live in `notebooks/`, the helper in `scripts/montecarlo.py`, and the jobs in
`jobs.json`.

## 1. Describe the workflow

`varying=["n_samples", "seed"]` matches the two levels in `jobs.json`. We keep
the default `python3` kernel so this runs locally with no setup.

In [ ]:
import nb2slurm

wf = nb2slurm.Workflow(
    name="monte_carlo_pi",
    notebooks=[
        "notebooks/0_settings.ipynb",
        "notebooks/1_simulate.ipynb",
        "notebooks/2_plot.ipynb",
    ],
    kernel="python3",
    varying=["n_samples", "seed"],
    jobs_json="jobs.json",
    resources=dict(nodes=1, cpus=1, time="00:05:00"),
)

## 2. The jobs

`jobs.json` already holds the grid; read it back to see the jobs it expands to.

In [ ]:
for job in wf.jobs_from_json():
    print(job)
# -> ('1000', '1') ('1000', '2') ('1000', '3') ('1000000', '1') ...

## 3. Generate the scripts

Renders `scripts/run_workflow.py`, `job.slurm`, the submitters and `jobs.txt`.

In [ ]:
for kind, path in wf.build().items():
    print(f"{kind:13s} -> {path}")

## 4. Test it locally

No special runner needed: just open the workflow notebooks and run them. In
Jupyter, open `notebooks/0_settings.ipynb`, then `1_simulate.ipynb`, then
`2_plot.ipynb`, and run each top to bottom (or **Run All**). Their parameters
cells hold sensible defaults (`n_samples=1000`, `seed=1`, a local `outdir`), so
they write `settings.json`, `result.json` and `scatter.png` you can inspect
straight away.

These are the *same* notebooks SLURM will execute — nb2slurm just injects a
different `(n_samples, seed)` per job and points `outdir` at
`output/<n_samples>/<seed>/` so jobs don't collide. If the chain works in
Jupyter it will work on the cluster. That's the duality: develop locally, scale
out unchanged.

## 5. Send it to the HPC and run there

The full cluster cycle is five calls — nb2slurm hides the ssh/rsync/sbatch:

1. `wf.push(ssh=cfg)` — sync your source **up** to the cluster (notebooks,
   scripts, `jobs.json`). Outputs are never uploaded.
2. `wf.check(ssh=cfg)` — preflight: remote dir, notebooks, built scripts, kernel.
3. `wf.submit(ssh=cfg)` — one SLURM job per `(n_samples, seed)`.
4. `wf.status(ssh=cfg)` — monitor the queue.
5. `wf.pull(ssh=cfg)` — bring `output/` (and the `done/` ledger) back **down**.

First describe your login node. Building the `SSHConfig` does no I/O, so this
cell is safe to run as-is; edit the values to match your account.

In [ ]:
cfg = nb2slurm.SSHConfig(
    host="spider.surf.nl",
    user="me",
    remote_dir="/home/me/monte_carlo_pi",
    key_filename="~/.ssh/id_ed25519",
)

### Preview the sync and the submit (dry runs)

`dry_run=True` prints the exact `rsync` / `sbatch` commands without touching the
network, so you can see precisely what will be sent and run — no cluster needed.

`push` syncs the project **up**, excluding `output/`, `done/`, `.git` and
caches, so re-pushing notebook edits can never clobber results already on the
cluster.

In [ ]:
wf.push(ssh=cfg, dry_run=True)

`submit` issues one throttled `sbatch` per subset (job N waits for job
N‑`concurrency`, so at most a few run at once).

In [ ]:
wf.submit(ssh=cfg, dry_run=True)

### The live run

Once `cfg` points at your real account, drop `dry_run` and run the full cycle.
(Left commented so this notebook stays runnable without a cluster.)

In [ ]:
# wf.push(ssh=cfg)        # 1. upload source up to remote_dir (never output/)
# wf.check(ssh=cfg)       # 2. preflight: dir, notebooks, scripts, kernel
# wf.submit(ssh=cfg)      # 3. one SLURM job per (n_samples, seed)
# wf.status(ssh=cfg)      # 4. monitor the queue (re-run until it empties)
# wf.pull(ssh=cfg)        # 5. bring output/ (+ done/) back down

## 6. Analyse the results

After `wf.pull(...)` brings every subset's `output/<n_samples>/<seed>/` back,
open `analyse_subsets.ipynb` to compare them all — π estimate per seed and the
$1/\sqrt{n}$ convergence across sample sizes.

## What made this nb2slurm-compatible?

* `0_settings.ipynb` has a `parameters` cell (`n_samples`, `seed`, `outdir`) and
  writes `settings.json`.
* `1_simulate` / `2_plot` have a `settings_path` parameters cell and load it.
* All outputs are written under `settings["outdir"]`, so parallel jobs don't
  collide.
* The simulate step is idempotent (skips if `result.json` exists).
* The helper is imported from `scripts/` with a `sys.path` fallback.
* The plot uses a headless backend + `display()`.

See `docs/setup_notebooks.ipynb` for the general checklist.